# RetailHero · A/B-тест и uplift-моделирование

**X5 Retail Group | Python · A/B Testing · Machine Learning**

Цель исследования — оценить эффект рекламной коммуникации на конверсию в покупку и проверить, можно ли точнее выбирать аудиторию для рассылки.

| Результат | Значение |
|:--|--:|
| Клиентов в эксперименте | **200 039** |
| Контроль / коммуникация | **60,33% / 63,65%** |
| Средний эффект | **+3,32 п. п.** |
| Uplift в верхних 10% по ML-модели (holdout) | **+8,36 п. п.** |

*Результаты приведены для исходного датасета; ниже все расчёты воспроизводятся кодом.*

## 1. Данные

Используются два файла RetailHero: участники эксперимента с признаком коммуникации (`treatment_flg`) и фактом покупки (`target`), а также данные о клиентах. История покупок подключается отдельно в виде агрегатов на клиента.

In [ ]:
%pip -q install huggingface_hub pandas numpy scipy statsmodels scikit-learn matplotlib

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chisquare
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
import statsmodels.formula.api as smf
from huggingface_hub import hf_hub_download

pd.set_option('display.float_format', '{:.3f}'.format)
plt.rcParams.update({
    'figure.figsize': (9, 4), 'figure.dpi': 125,
    'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False
})
GREEN, GRAY = '#147d74', '#9aa9b5'

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
REPO = 'pytorch-lifestream/retailhero-uplift'


def load_file(name):
    local = DATA_DIR / name
    if not local.exists():
        local = Path(hf_hub_download(
            repo_id=REPO, filename=f'data/{name}', repo_type='dataset'
        ))
    return pd.read_csv(local, dtype={'client_id': 'string'})

ab = load_file('uplift_train.csv.gz')
clients = load_file('clients.csv.gz')

data = ab.merge(clients, on='client_id', how='left', validate='one_to_one')
assert data['client_id'].is_unique
assert data['treatment_flg'].isin([0, 1]).all()
assert data['target'].isin([0, 1]).all()

print('Клиентов:', len(data))
display(data[['treatment_flg', 'target', 'age', 'gender']].head())

## 2. A/B-тест

**H₀:** коммуникация не влияет на конверсию.  
**H₁:** конверсия между группами различается.

Основная метрика — доля клиентов с `target = 1`. Для сравнения используем z-тест двух долей и 95% доверительный интервал разности.

In [ ]:
summary = data.groupby('treatment_flg')['target'].agg(['size', 'sum', 'mean'])
summary.columns = ['clients', 'buyers', 'conversion']

n0, n1 = summary['clients'].to_numpy()
b0, b1 = summary['buyers'].to_numpy()
p0, p1 = summary['conversion'].to_numpy()

effect = p1 - p0
se = np.sqrt(p0 * (1 - p0) / n0 + p1 * (1 - p1) / n1)
ci = (effect - 1.96 * se, effect + 1.96 * se)
z, p_value = proportions_ztest([b1, b0], [n1, n0])

result = summary.copy()
result.index = ['Контроль', 'Коммуникация']
result['conversion'] *= 100
display(result.round(2))

print(f'Эффект: {100 * effect:+.2f} п. п.')
print(f'95% ДИ: [{100 * ci[0]:+.2f}; {100 * ci[1]:+.2f}] п. п.')
print(f'z = {z:.2f}; p-value = {p_value:.2e}')

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
vals = [100 * p0, 100 * p1]
bars = ax.bar(['Контроль', 'Коммуникация'], vals,
              color=[GRAY, GREEN], width=0.48)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.5,
            f'{val:.2f}%', ha='center', fontweight='bold')
ax.set_ylim(0, 76)
ax.set_ylabel('Конверсия, %')
ax.set_title('Эффект рекламной коммуникации', loc='left', weight='bold')
ax.grid(axis='y', alpha=0.13)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

Коммуникация повысила конверсию на **3,32 п. п.** (95% ДИ: **2,90–3,75 п. п.**). Разница статистически значима. Это примерно 33 дополнительных покупателя на 1 000 контактов при переносимости эффекта на аналогичную аудиторию.

## 3. Проверки эксперимента

Проверяем соотношение размеров групп (SRM), баланс доступных характеристик и аномальные значения возраста.

In [ ]:
chi2, p_srm = chisquare([n0, n1], [len(data) / 2] * 2)

data['age_clean'] = pd.to_numeric(data['age'], errors='coerce')
data.loc[~data['age_clean'].between(18, 100), 'age_clean'] = np.nan

data['age_group'] = pd.cut(
    data['age_clean'], bins=[17, 29, 44, 59, 74, 100],
    labels=['18–29', '30–44', '45–59', '60–74', '75–100']
).astype('object').fillna('Unknown')
data['gender_group'] = data['gender'].fillna('U')

print(f'SRM p-value: {p_srm:.3f}')
print('Неизвестный/аномальный возраст:', data['age_clean'].isna().sum())
print('Средний возраст (A/B):')
display(data.groupby('treatment_flg')['age_clean'].mean().round(2))
print('Пол, % по группам:')
display((pd.crosstab(data.treatment_flg, data.gender_group,
                     normalize='index') * 100).round(2))

SRM не выявил отклонения от распределения 50/50 (**p = 0,863**). Возраст и пол распределены между группами близко. Некорректные значения возраста исключены из числового анализа, но сами клиенты оставлены в выборке.

## 4. История покупок и сегменты

Для оценки покупательской активности используются исторические данные до коммуникации: число чеков и сумма расходов. В основной ноутбук загружается уже агрегированный файл `final_stats.csv` (`client_id`, `count`, `sum`).

In [ ]:
feature_file = next((p for p in [Path('final_stats.csv'), DATA_DIR / 'final_stats.csv']
                     if p.exists()), None)
has_history = feature_file is not None

if has_history:
    history = pd.read_csv(feature_file, dtype={'client_id': 'string'})
    history = history[['client_id', 'count', 'sum']].rename(
        columns={'count': 'pre_checks', 'sum': 'pre_spending'}
    )
    data = data.merge(history, on='client_id', how='left', validate='one_to_one')
    print('История покупок подключена:', feature_file)
    display(data.groupby('treatment_flg')[['pre_checks', 'pre_spending']].mean().round(2))
else:
    print('История покупок не найдена — доступны демографические признаки.')

In [ ]:
segments = ['age_group', 'gender_group']

if has_history:
    for col, name in [('pre_checks', 'checks_group'),
                      ('pre_spending', 'spending_group')]:
        quartile = pd.qcut(data[col], 4, labels=False, duplicates='drop')
        data[name] = quartile.map(
            lambda x: f'Q{int(x) + 1}' if pd.notna(x) else 'Unknown'
        )
        segments.append(name)


def compare_segments(column):
    rows = []
    for name, group in data.groupby(column):
        a = group.loc[group.treatment_flg.eq(0), 'target']
        b = group.loc[group.treatment_flg.eq(1), 'target']
        if len(a) < 30 or len(b) < 30:
            continue
        pa, pb = a.mean(), b.mean()
        d = pb - pa
        err = np.sqrt(pa * (1 - pa) / len(a) + pb * (1 - pb) / len(b))
        _, p = proportions_ztest([b.sum(), a.sum()], [len(b), len(a)])
        rows.append([column, str(name), len(a), len(b),
                     100 * pa, 100 * pb, 100 * d,
                     100 * (d - 1.96 * err), 100 * (d + 1.96 * err), p])
    return pd.DataFrame(rows, columns=[
        'dimension', 'segment', 'n_control', 'n_treatment',
        'control_%', 'treatment_%', 'uplift_pp', 'ci_low', 'ci_high', 'p_value'
    ])

segment_results = pd.concat([compare_segments(s) for s in segments], ignore_index=True)
segment_results['p_fdr'] = multipletests(segment_results.p_value, method='fdr_bh')[1]
display(segment_results.round(3))

In [ ]:
labels = {
    'age_group': 'Возраст', 'gender_group': 'Пол',
    'checks_group': 'Частота покупок', 'spending_group': 'Расходы'
}
fig, axes = plt.subplots(len(segments), 1,
                         figsize=(9, len(segments) * 2.8), squeeze=False)

for ax, col in zip(axes.ravel(), segments):
    part = segment_results.loc[segment_results.dimension.eq(col)].copy()
    if col == 'age_group':
        order = ['18–29', '30–44', '45–59', '60–74', '75–100', 'Unknown']
        part['rank'] = part.segment.map({x: i for i, x in enumerate(order)})
        part = part.sort_values('rank')
    else:
        part = part.sort_values('segment')

    x = part.uplift_pp.to_numpy()
    errors = [x - part.ci_low.to_numpy(), part.ci_high.to_numpy() - x]
    y = np.arange(len(part))
    ax.errorbar(x, y, xerr=errors, fmt='o', color=GREEN,
                capsize=3, markersize=6)
    ax.axvline(0, linestyle='--', color=GRAY, linewidth=1)
    ax.set_yticks(y, part.segment)
    ax.invert_yaxis()
    ax.set_xlabel('Эффект, п. п. (95% ДИ)')
    ax.set_title(labels[col], loc='left', weight='bold')
    ax.grid(axis='x', alpha=0.12)

plt.tight_layout()
plt.show()

По результатам сегментации больший отклик наблюдается у части старших клиентов и покупателей с меньшими историческими расходами. Различия отдельных эффектов проверяем отдельно, а не только по значимости внутри групп.

In [ ]:
interaction_rows = []
for col in segments:
    model = smf.ols(f'target ~ treatment_flg * C({col})', data=data).fit(cov_type='HC3')
    terms = [k for k in model.params.index if k.startswith('treatment_flg:C(')]
    if not terms:
        continue
    r = np.zeros((len(terms), len(model.params)))
    for i, term in enumerate(terms):
        r[i, model.params.index.get_loc(term)] = 1
    test = model.wald_test(r, scalar=True)
    interaction_rows.append([col, float(test.statistic), float(test.pvalue)])

interaction_results = pd.DataFrame(
    interaction_rows, columns=['segment', 'wald_stat', 'p_value']
)
interaction_results['p_fdr'] = multipletests(
    interaction_results.p_value, method='fdr_bh'
)[1]
display(interaction_results.round(4))

## 5. Uplift-модель

Для персонализации коммуникации используем **T-learner** — две модели градиентного бустинга, обученные отдельно на контроле и эксперименте. Разница их прогнозов даёт оценку дополнительной вероятности покупки при коммуникации.

Качество проверяем на отложенных **30% клиентов**, которые не использовались при обучении.

In [ ]:
ml = data.copy()
ml['age_unknown'] = ml.age_clean.isna().astype(int)
ml['age_value'] = ml.age_clean.fillna(ml.age_clean.median())

x_cols = ['age_value', 'age_unknown']
gender = pd.get_dummies(ml.gender_group, prefix='gender')
ml = pd.concat([ml, gender], axis=1)
x_cols += gender.columns.tolist()

if has_history:
    for col in ['pre_checks', 'pre_spending']:
        ml[f'{col}_missing'] = ml[col].isna().astype(int)
        ml[col] = ml[col].fillna(ml[col].median())
        x_cols += [col, f'{col}_missing']

X = ml[x_cols].astype(float)
y = ml.target.astype(int)
t = ml.treatment_flg.astype(int)
strata = t.astype(str) + '_' + y.astype(str)

idx_train, idx_test = train_test_split(
    ml.index, test_size=0.30, stratify=strata, random_state=42
)

models = {}
for treatment in [0, 1]:
    train_subset = idx_train[t.loc[idx_train].eq(treatment).to_numpy()]
    clf = HistGradientBoostingClassifier(
        learning_rate=0.05, max_depth=4, max_iter=250,
        min_samples_leaf=80, random_state=42
    )
    clf.fit(X.loc[train_subset], y.loc[train_subset])
    models[treatment] = clf

pred0 = models[0].predict_proba(X.loc[idx_test])[:, 1]
pred1 = models[1].predict_proba(X.loc[idx_test])[:, 1]

holdout = ml.loc[idx_test, ['treatment_flg', 'target']].copy()
holdout['uplift_score'] = pred1 - pred0
holdout = holdout.sort_values('uplift_score', ascending=False).reset_index(drop=True)

print('Train:', len(idx_train), '| Holdout:', len(idx_test))
print('Признаки:', ', '.join(x_cols))

In [ ]:
def observed_uplift(frame):
    rates = frame.groupby('treatment_flg').target.mean()
    return rates.loc[1] - rates.loc[0]

baseline = observed_uplift(holdout)
rows = []
for share in [0.1, 0.2, 0.3, 0.5, 1.0]:
    top = holdout.head(max(1, int(len(holdout) * share)))
    rows.append([share, len(top), 100 * observed_uplift(top)])

topk = pd.DataFrame(rows, columns=['share', 'clients', 'uplift_pp'])

holdout['decile'] = pd.qcut(
    holdout.uplift_score.rank(method='first', ascending=False),
    10, labels=[f'D{i}' for i in range(1, 11)]
)
decile_rows = []
for name, group in holdout.groupby('decile', observed=True):
    decile_rows.append([str(name), len(group),
                        group.uplift_score.mean(), 100 * observed_uplift(group)])
deciles = pd.DataFrame(decile_rows, columns=['decile', 'clients', 'score', 'uplift_pp'])

print(f'Средний uplift на holdout: {100 * baseline:.2f} п. п.')
display(topk.round(3))
display(deciles.round(3))

In [ ]:
curve = []
for share in np.linspace(0.1, 1.0, 10):
    top = holdout.head(max(1, int(len(holdout) * share)))
    curve.append([share, 100 * observed_uplift(top),
                  len(top) * observed_uplift(top)])
curve = pd.DataFrame(curve, columns=['share', 'uplift_pp', 'extra_buyers'])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(curve.share * 100, curve.uplift_pp, '-o', color=GREEN)
ax[0].axhline(100 * baseline, linestyle='--', color=GRAY, label='Средний эффект')
ax[0].set(xlabel='Топ клиентов, %', ylabel='Uplift, п. п.',
          title='Фактический эффект по выбранной аудитории')
ax[0].legend(frameon=False)

ax[1].plot(curve.share * 100, curve.extra_buyers, '-o', color=GREEN,
           label='T-learner')
ax[1].plot(curve.share * 100, len(holdout) * curve.share * baseline,
           '--', color=GRAY, label='Случайный выбор (ожидание)')
ax[1].set(xlabel='Топ клиентов, %', ylabel='Дополнительные покупатели',
          title='Накопленный эффект на holdout')
ax[1].legend(frameon=False)
for a in ax:
    a.grid(alpha=0.12)
plt.tight_layout()
plt.show()

## 6. Выводы

- Рекламная коммуникация увеличила конверсию с **60,33% до 63,65%**; оценка эффекта — **+3,32 п. п.**
- Отклик клиентов неоднороден: наибольшие абсолютные эффекты наблюдаются в ряде возрастных и поведенческих сегментов.
- T-learner выделил верхние **10% аудитории** с фактическим uplift **+8,36 п. п.** на holdout против **+3,32 п. п.** в среднем.
- Результат показывает потенциал персонализированного таргетинга, но требует проверки на новых кампаниях. Данных о стоимости контакта и марже недостаточно для оценки ROI.

*Ограничения: открытые данные не позволяют полностью проверить доставку сообщений и рандомизацию; результаты по сегментам и uplift-модели носят исследовательский характер.*

In [ ]:
out = Path('outputs')
out.mkdir(exist_ok=True)
summary.to_csv(out / 'ab_results.csv')
segment_results.to_csv(out / 'segments.csv', index=False)
interaction_results.to_csv(out / 'interactions.csv', index=False)
topk.to_csv(out / 'uplift_topk.csv', index=False)
deciles.to_csv(out / 'uplift_deciles.csv', index=False)
print('Результаты сохранены в outputs/')

---

**Данные:** [ODS RetailHero](https://ods.ai/competitions/x5-retailhero-uplift-modeling/data) · [Hugging Face](https://huggingface.co/datasets/pytorch-lifestream/retailhero-uplift/tree/main/data)